In [ ]:
import openai
import pandas as pd
import json
import requests


from google.colab import userdata
openai_key= userdata.get('OPENAI_API_KEY')
openai.api_key = openai_key

def extract_email_from_domain(domain):
    prompt = f"""
    You are an email extraction assistant.
    Task:
    - Extract the **primary contact email** from the website: {domain}
    - If no email exists, provide a **parent/official/general email** pattern.

    Respond ONLY in JSON with the format:
    {{
      "domain": "{domain}",
      "primary_email": "<primary or null>",
      "secondary_email": "<secondary or null>"
    }}
    """

    try:
        response = openai.ChatCompletion.create(
            model="gpt-4",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        content = response["choices"][0]["message"]["content"].strip()
        return json.loads(content)

    except Exception as e:
        return {
            "domain": domain,
            "primary_email": None,
            "secondary_email": None,
            "error": str(e)
        }


def process_excel(input_file, output_file):
    # Load Excel
    df = pd.read_excel(input_file)

    results = []
    for idx, row in df.iterrows():
        domain = row["Domain"]

        data = extract_email_from_domain(domain)
        results.append(data)

        # Print for verification
        print(f'{data["domain"]} | Primary: {data["primary_email"]} | Secondary: {data["secondary_email"]}')

    # Convert results to DataFrame
    results_df = pd.DataFrame(results)
    results_df.to_csv(output_file, index=False)
    print(f"\n✅ Results saved to {output_file}")


# Example usage
process_excel("/content/missing email.xlsx", "domain_emails.csv")


apxpcs.com | Primary: None | Secondary: None
apxton.com | Primary: None | Secondary: None
apyapy.com | Primary: None | Secondary: None
aq19hs.us | Primary: None | Secondary: None
aq9jtd.us | Primary: None | Secondary: None
aqaire.com | Primary: None | Secondary: None
aqbags.com | Primary: None | Secondary: None
aqclux.com | Primary: None | Secondary: None
aqcups.com | Primary: None | Secondary: None

✅ Results saved to domain_emails.csv


In [ ]:
import openai
from openai import OpenAI


from google.colab import userdata
openai_key= userdata.get('OPENAI_API_KEY')


client = OpenAI(api_key=openai_key)

def get_email_from_gpt(domain: str):
    system_prompt = (
        "You are an assistant that suggests official contact emails for websites. "
        "If no email is provided, Check for it's parent website email. "
        "Respond ONLY in JSON format with keys: domain, primary_email, secondary_email."
    )

    user_prompt = f"Domain: {domain}"

    try:
        response = client.chat.completions.create(
            model="gpt-4",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {e}"


test_domain = "fitoys.com"
print(get_email_from_gpt(test_domain))


{
  "domain": "fitoys.com",
  "primary_email": "info@fitoys.com",
  "secondary_email": "support@fitoys.com"
}


**Chatgpt is hallusinating and randomly guessing the emails.**

In [ ]:
import requests
from openai import OpenAI

from google.colab import userdata
openai_key= userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=openai_key)


SYSTEM_PROMPT = ("You are an assistant that extracts contact emails only from the provided website content. "
                 "Search mailto links, visible emails, JSON-LD, schema.org ContactPoint, and meta tags. "
                 "Do NOT invent or guess. Output ONLY JSON with keys: domain, primary_email, secondary_email.")

def fetch_html(domain):
    urls = [f"https://{domain}", f"http://{domain}", f"https://www.{domain}", f"http://www.{domain}"]
    headers = {"User-Agent":"Mozilla/5.0"}
    for u in urls:
        try:
            r = requests.get(u, headers=headers, timeout=10)
            if r.status_code == 200 and r.text:
                return r.text
        except requests.RequestException:
            continue
    return ""

def extract_email_via_gpt(domain):
    html = fetch_html(domain)
    if not html:
        return {"domain": domain, "primary_email": None, "secondary_email": None}
    messages = [
        {"role":"system","content":SYSTEM_PROMPT},
        {"role":"user","content": f"Domain: {domain}\n\nHTML:\n{html[:50000]}"}  # truncate if huge
    ]
    resp = client.chat.completions.create(model="gpt-4", messages=messages, temperature=0, max_tokens=400)
    out = resp.choices[0].message.content.strip()
    try:
        return json.loads(out)
    except:
        # fallback to regex local extraction (guaranteed)
        emails = re.findall(r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[A-Za-z]{2,}", html)
        emails = sorted(set(emails))
        return {"domain": domain, "primary_email": emails[0] if emails else None, "secondary_email": emails[1] if len(emails)>1 else None}

# Example:
print(extract_email_via_gpt("aqliah.com"))


RateLimitError: Error code: 429 - {'error': {'message': 'Request too large for gpt-4 in organization org-UVoyNd5JJTn1UkdxzufjS6fE on tokens per min (TPM): Limit 10000, Requested 12709. The input or output tokens must be reduced in order to run successfully. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}

**Chatgpt have a limited context window so its token limit exceed because the websites have a too much data so chatgpt can not perform any operation on it because of token limit.**

In [ ]:
import requests, re, json
from openai import OpenAI

from google.colab import userdata
openai_key= userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=openai_key)

SYSTEM_PROMPT = (
    "You are an assistant that extracts contact emails only from the provided website content. "
    "Search mailto links, visible emails, JSON-LD, schema.org ContactPoint, and meta tags. "
    "Do NOT invent or guess. If no email is present, return primary_email:null and secondary_email:null. "
    "Output ONLY JSON with keys: domain, primary_email, secondary_email."
)

def fetch_html(domain, max_chars=15000):
    """Fetch HTML and truncate to avoid token overflow."""
    urls = [
        f"https://{domain}",
        f"http://{domain}",
        f"https://www.{domain}",
        f"http://www.{domain}",
    ]
    headers = {"User-Agent": "Mozilla/5.0"}
    for u in urls:
        try:
            r = requests.get(u, headers=headers, timeout=10)
            if r.status_code == 200 and r.text:
                return r.text[:max_chars]  # truncate to safe size
        except requests.RequestException:
            continue
    return ""

def extract_email_via_gpt(domain):
    html = fetch_html(domain)
    if not html:
        return {"domain": domain, "primary_email": None, "secondary_email": None}

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Domain: {domain}\n\nHTML:\n{html}"}
    ]

    try:
        resp = client.chat.completions.create(
            model="gpt-4",
            messages=messages,
            temperature=0,
            max_tokens=400
        )
        out = resp.choices[0].message.content.strip()
        return json.loads(out)
    except Exception as e:
        # fallback regex if JSON fails
        emails = re.findall(r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[A-Za-z]{2,}", html)
        emails = sorted(set(emails))
        return {
            "domain": domain,
            "primary_email": emails[0] if emails else None,
            "secondary_email": emails[1] if len(emails) > 1 else None,
            "note": f"fallback due to {e}"
        }

# Example:
print(extract_email_via_gpt("fitpro.club"))


{'domain': 'fitpro.club', 'primary_email': None, 'secondary_email': None}
